<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# Lab: Train and Evaluate an RNN-Based Classifier in PyTorch


Estimated time needed: **30** minutes


## Learning Objectives

After completing this lab, you will be able to:

<ul>
    <li> Prepare and preprocess text data for training an RNN-based sentiment classifier.</li>
    <li> Implement an RNN model architecture using PyTorch modules.</li><li> Train the model using appropriate loss functions and optimizers</li>
    <li> Evaluate model performance using metrics such as accuracy and loss.</li><li> Identify training behaviors such as overfitting and convergence trends.</li>
</ul> 


## Table of Contents

<ol>
<li><a href="#Import-Required-Libraries">Import Required Libraries</a> </li>
<li><a href="#Recurrent-Neural-Networks-for-Text-Classification">Recurrent Neural Networks for Text Classification</a> </li>
<li><a href="#Create-Sentiment-Data">Create Sentiment Data</a> </li>
<li><a href="#Preprocess-Text-Data">Preprocess Text Data</a> </li>
<li><a href="#Create-Dataset-and-DataLoader">Create Dataset and DataLoader</a> </li>
<li><a href="#Implement-an-RNN-Classifier">Implement an RNN Classifier</a> </li>
<li><a href="#Train-the-Classifier">Train the Classifier</a> </li>
<li><a href="#Evaluate-Accuracy-and-Loss">Evaluate Accuracy and Loss</a> </li>
<li><a href="#Identify-Training-Behaviors">Identify Training Behaviors</a> </li>

</ol>


----


## Import Required Libraries


The following required libraries are __not__ pre-installed in the Skills Network Labs environment. __You will need to run the following cell__ to install them:


In [1]:
!pip install torch

In [2]:
import random
import re
from collections import Counter

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import Dataset, DataLoader, random_split

random.seed(2)
torch.manual_seed(2)

## Recurrent Neural Networks for Text Classification


A recurrent neural network processes a sequence one step at a time. For text classification, each token is passed through the recurrent layer, and the final hidden state is used to predict the class.


Unlike feedforward models, an RNN reuses hidden state information as it reads the sequence. This makes it useful for ordered text data, but simple RNNs can struggle with long-range dependencies.
<p style="text-align:center">
    <img src="https://commons.wikimedia.org/wiki/Special:Redirect/file/Recurrent%20neural%20network%20unfold.svg" width="650" alt="Unfolded recurrent neural network architecture diagram">
</p>
<p style="text-align:center"><em>Source: Wikimedia Commons, Recurrent neural network unfold.svg</em></p>


## Create Sentiment Data


Create a small sentiment dataset. The label <code>1</code> represents positive sentiment, and the label <code>0</code> represents negative sentiment.


In [3]:
# Create a sentiment dataset

positive_sentences = [
    "the product is reliable and easy to use",
    "the support team was helpful and quick",
    "the lesson was clear and practical",
    "the app runs smoothly and looks clean",
    "the package arrived early and intact",
    "the headphones sound rich and balanced",
    "the interface is simple and polished",
    "the service was friendly and efficient",
    "the device feels durable and fast",
    "the course examples were useful and engaging",
]

negative_sentences = [
    "the product is fragile and hard to use",
    "the support team was slow and unhelpful",
    "the lesson was unclear and confusing",
    "the app crashes often and looks messy",
    "the package arrived late and damaged",
    "the headphones sound harsh and distorted",
    "the interface is awkward and frustrating",
    "the service was rude and inefficient",
    "the device feels cheap and slow",
    "the course examples were vague and outdated",
]

data = [(sentence, 1) for sentence in positive_sentences] + [(sentence, 0) for sentence in negative_sentences]
random.shuffle(data)

print("Number of examples:", len(data))
print(data[:3])

Number of examples: 20
[('the service was friendly and efficient', 1), ('the interface is simple and polished', 1), ('the service was rude and inefficient', 0)]


Add simple variations to increase the number of training examples:


In [4]:
# Expand the dataset with short prefixes

prefixes = ["", "overall ", "today ", "in my experience "]
expanded_data = []

for sentence, label in data:
    for prefix in prefixes:
        expanded_data.append((prefix + sentence, label))

random.shuffle(expanded_data)

print("Expanded examples:", len(expanded_data))
print(expanded_data[:3])

Expanded examples: 80
[('the service was friendly and efficient', 1), ('today the lesson was clear and practical', 1), ('the headphones sound rich and balanced', 1)]


## Preprocess Text Data


Preprocessing converts each sentence into tokens and then into numerical token IDs that can be passed to an embedding layer.


In [5]:
# Create a tokenizer

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

Tokenize the dataset and build a vocabulary:


In [6]:
# Tokenize examples and build vocabulary

tokenized_examples = [(tokenize(text), label) for text, label in expanded_data]
token_counts = Counter(token for tokens, label in tokenized_examples for token in tokens)

special_tokens = ["<pad>", "<unk>"]
vocab = {token: index for index, token in enumerate(special_tokens)}

for token in token_counts:
    if token not in vocab:
        vocab[token] = len(vocab)

pad_id = vocab["<pad>"]
unk_id = vocab["<unk>"]

print("Vocabulary size:", len(vocab))
print(list(vocab.items())[:10])

Vocabulary size: 71
[('<pad>', 0), ('<unk>', 1), ('the', 2), ('service', 3), ('was', 4), ('friendly', 5), ('and', 6), ('efficient', 7), ('today', 8), ('lesson', 9)]


Encode tokens as token IDs:


In [7]:
# Convert tokens to token IDs

def encode_tokens(tokens, vocabulary):
    return [vocabulary.get(token, unk_id) for token in tokens]

encoded_examples = [(encode_tokens(tokens, vocab), label) for tokens, label in tokenized_examples]

print(tokenized_examples[0])
print(encoded_examples[0])

(['the', 'service', 'was', 'friendly', 'and', 'efficient'], 1)
([2, 3, 4, 5, 6, 7], 1)


## Create Dataset and DataLoader


Create a custom Dataset and use a DataLoader to batch variable-length sequences with padding.


In [8]:
# Create a custom Dataset

class SentimentDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        token_ids, label = self.examples[index]
        return torch.tensor(token_ids, dtype=torch.long), torch.tensor(label, dtype=torch.float32)


full_dataset = SentimentDataset(encoded_examples)

print("Dataset size:", len(full_dataset))
print(full_dataset[0])

Dataset size: 80
(tensor([2, 3, 4, 5, 6, 7]), tensor(1.))


Split the data into training and validation sets:


In [9]:
# Split the dataset

train_size = int(0.8 * len(full_dataset))
valid_size = len(full_dataset) - train_size

train_dataset, valid_dataset = random_split(
    full_dataset,
    [train_size, valid_size],
    generator=torch.Generator().manual_seed(2),
)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(valid_dataset))

Training examples: 64
Validation examples: 16


Create a collate function to pad sequences in each batch:


In [10]:
# Create a collate function

def collate_batch(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(sequence) for sequence in sequences], dtype=torch.long)
    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=pad_id)
    labels = torch.stack(labels)
    return padded_sequences, labels, lengths

Create DataLoaders:


In [11]:
# Create training and validation DataLoaders

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_batch)
valid_loader = DataLoader(valid_dataset, batch_size=8, shuffle=False, collate_fn=collate_batch)

batch_sequences, batch_labels, batch_lengths = next(iter(train_loader))

print("Batch sequences shape:", batch_sequences.shape)
print("Batch labels shape:", batch_labels.shape)
print("Batch lengths:", batch_lengths)

Batch sequences shape: torch.Size([8, 11])
Batch labels shape: torch.Size([8])
Batch lengths: tensor([ 6,  7,  8,  9,  9,  8,  7, 11])


## Implement an RNN Classifier


The classifier uses an embedding layer, a recurrent layer, and a linear output layer for binary classification.


In [12]:
# Create an RNN classifier

class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, padding_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, sequences, lengths):
        embedded = self.embedding(sequences)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, hidden = self.rnn(packed)
        logits = self.fc(hidden[-1])
        return logits.squeeze(1)

Instantiate the RNN classifier and check the output shape:


In [13]:
# Instantiate the RNN classifier

model = RNNClassifier(
    vocab_size=len(vocab),
    embedding_dim=32,
    hidden_dim=32,
    output_dim=1,
    padding_idx=pad_id,
)

sample_logits = model(batch_sequences, batch_lengths)

print(model)
print("Output shape:", sample_logits.shape)

RNNClassifier(
  (embedding): Embedding(71, 32, padding_idx=0)
  (rnn): RNN(32, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)
Output shape: torch.Size([8])


## Train the Classifier


Use binary cross-entropy with logits as the loss function and Adam as the optimizer.


In [14]:
# Define the loss function and optimizer

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

Create helper functions for training and evaluation:


In [15]:
# Create training and evaluation helpers

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for sequences, labels, lengths in loader:
        optimizer.zero_grad()
        logits = model(sequences, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for sequences, labels, lengths in loader:
            logits = model(sequences, lengths)
            loss = criterion(logits, labels)
            predictions = (torch.sigmoid(logits) >= 0.5).float()
            total_loss += loss.item()
            correct += (predictions == labels).sum().item()
            total += labels.numel()
    return total_loss / len(loader), correct / total


Train the RNN classifier and track loss and accuracy:


In [16]:
# Train the RNN classifier

history = []
epochs = 8

for epoch in range(epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    valid_loss, valid_accuracy = evaluate_model(model, valid_loader, criterion)
    history.append({"epoch": epoch + 1, "train_loss": train_loss, "valid_loss": valid_loss, "valid_accuracy": valid_accuracy})
    print(f"Epoch {epoch + 1}: train loss={train_loss:.4f}, valid loss={valid_loss:.4f}, valid accuracy={valid_accuracy:.4f}")

Epoch 1: train loss=0.6771, valid loss=0.5483, valid accuracy=0.6250
Epoch 2: train loss=0.4295, valid loss=0.3315, valid accuracy=0.9375
Epoch 3: train loss=0.2183, valid loss=0.1148, valid accuracy=1.0000
Epoch 4: train loss=0.0639, valid loss=0.0266, valid accuracy=1.0000
Epoch 5: train loss=0.0161, valid loss=0.0107, valid accuracy=1.0000
Epoch 6: train loss=0.0059, valid loss=0.0047, valid accuracy=1.0000
Epoch 7: train loss=0.0031, valid loss=0.0029, valid accuracy=1.0000
Epoch 8: train loss=0.0021, valid loss=0.0021, valid accuracy=1.0000


## Evaluate Accuracy and Loss


After training, calculate validation loss and accuracy again as summary metrics.


In [17]:
# Evaluate the final model

final_valid_loss, final_valid_accuracy = evaluate_model(model, valid_loader, criterion)

print("Final validation loss:", round(final_valid_loss, 4))
print("Final validation accuracy:", round(final_valid_accuracy, 4))

Final validation loss: 0.0021
Final validation accuracy: 1.0


Collect predictions to inspect individual model outputs:


In [18]:
# Collect predictions and labels

model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for sequences, labels, lengths in valid_loader:
        logits = model(sequences, lengths)
        predictions = (torch.sigmoid(logits) >= 0.5).long()
        all_predictions.extend(predictions.tolist())
        all_labels.extend(labels.long().tolist())

print("Predictions:", all_predictions)
print("Labels:", all_labels)

Predictions: [1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1]
Labels: [1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1]


## Identify Training Behaviors


Training history can reveal convergence, underfitting, or overfitting. Compare training loss and validation loss across epochs.


In [19]:
# Inspect training history

for row in history:
    print(row)

{'epoch': 1, 'train_loss': 0.6770681589841843, 'valid_loss': 0.5482814311981201, 'valid_accuracy': 0.625}
{'epoch': 2, 'train_loss': 0.4294799752533436, 'valid_loss': 0.33147279918193817, 'valid_accuracy': 0.9375}
{'epoch': 3, 'train_loss': 0.21834664978086948, 'valid_loss': 0.11484763026237488, 'valid_accuracy': 1.0}
{'epoch': 4, 'train_loss': 0.06393421208485961, 'valid_loss': 0.026634030044078827, 'valid_accuracy': 1.0}
{'epoch': 5, 'train_loss': 0.016082072455901653, 'valid_loss': 0.010702125262469053, 'valid_accuracy': 1.0}
{'epoch': 6, 'train_loss': 0.005884075071662664, 'valid_loss': 0.004684706451371312, 'valid_accuracy': 1.0}
{'epoch': 7, 'train_loss': 0.0031041740876389667, 'valid_loss': 0.002888289513066411, 'valid_accuracy': 1.0}
{'epoch': 8, 'train_loss': 0.002081499798805453, 'valid_loss': 0.0021091459784656763, 'valid_accuracy': 1.0}


Use simple conditions to identify the training pattern:


In [20]:
# Interpret training behavior

first_train_loss = history[0]["train_loss"]
last_train_loss = history[-1]["train_loss"]
first_valid_loss = history[0]["valid_loss"]
last_valid_loss = history[-1]["valid_loss"]

if last_train_loss < first_train_loss and last_valid_loss <= first_valid_loss:
    print("The model shows convergence because training and validation loss improved.")
elif last_train_loss < first_train_loss and last_valid_loss > first_valid_loss:
    print("The model may be overfitting because validation loss increased while training loss decreased.")
else:
    print("The model may be underfitting because training loss did not improve enough.")

The model shows convergence because training and validation loss improved.


## Authors


**Harish Pant** is an accomplished IT Consultant specializing in bridging the gap between complex AI technologies and practical business applications. Holding a Bachelor’s degree in Computer Science with a strong background in Machine Learning, Harish serves clients across diverse industries—including fintech, retail, and manufacturing—helping them navigate AI adoption. Harish is known for translating technical AI concepts into actionable business strategies and delivering engaging, high-impact presentations.



<a href="https://dataplatform.cloud.ibm.com/registration/stepone?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork&context=cpdaas&apps=data_science_experience%2Cwatson_machine_learning"><img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork/Template/module%201/images/Watson_Studio.png"></a>


<!--## Change Log--


<!--|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2026-05-27|0.1|P.Kravitz|Innitial version|
|2026-06-03|0.2|Leah Hanson|QA review - Minor changes for grammar and flow|
|2026-06-08|0.3|P.Kravitz |Remove 2 unneeded lines and comment out the change log-->


## <h3 align="center"> &#169; IBM Corporation. All rights reserved. <h3/>
